# Stream C - Evidence Builder

Builds an offline, reproducible evidence table from processed manifest metadata.


In [1]:
from pathlib import Path
import sys, os, json, pickle, re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import STREAM_C_CACHE, STREAM_A_CACHE, STREAM_B_CACHE, ALL_MANIFEST, FIGURES, TABLES

STREAM_C_CACHE.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("STREAM_C_CACHE:", STREAM_C_CACHE)


ROOT: E:\eve-rag
STREAM_C_CACHE: E:\eve-rag\cache\stream_c


In [2]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
BATCH_SIZE = 256


Torch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM GB: 8.0


In [3]:
EVIDENCE_DIR = STREAM_C_CACHE / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(ALL_MANIFEST, dtype={"pair_id":"string","dataset":"string","split":"string","caption":"string","title":"string","topic":"string","source":"string","timestamp":"string","raw_image_path":"string","text_source":"string","image_source":"string","fake_cls":"string"}, low_memory=False)
print("Rows:", len(df))
display(df.groupby(["dataset", "split"]).size().reset_index(name="rows"))


Rows: 97361


,dataset,split,rows
0,mmfakebench,test,10000
1,mmfakebench,val,1000
2,newsclippings,test,7264
3,newsclippings,train,71072
4,newsclippings,val,7024
5,verite,test,1001


In [4]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def build_evidence_text(row):
    parts = []
    cap = clean_text(row.get("caption", ""))
    if cap:
        parts.append(f"Caption: {cap}")
    title = clean_text(row.get("title", ""))
    if title:
        parts.append(f"Article title: {title}")
    topic = clean_text(row.get("topic", "")); source = clean_text(row.get("source", "")); timestamp = clean_text(row.get("timestamp", ""))
    if topic or source or timestamp:
        parts.append(f"News metadata: topic={topic}; source={source}; timestamp={timestamp}")
    text_source = clean_text(row.get("text_source", "")); image_source = clean_text(row.get("image_source", "")); fake_cls = clean_text(row.get("fake_cls", ""))
    if text_source or image_source or fake_cls:
        parts.append(f"MMFakeBench metadata: text_source={text_source}; image_source={image_source}; fake_cls={fake_cls}")
    raw_img = clean_text(row.get("raw_image_path", ""))
    if raw_img:
        parts.append(f"Raw image reference: {raw_img}")
    parts.append(f"Dataset record: dataset={clean_text(row.get('dataset',''))}; split={clean_text(row.get('split',''))}")
    return " | ".join(parts)

evidence = df[["pair_id", "dataset", "split", "label", "caption"]].copy()
evidence["evidence_text"] = df.apply(build_evidence_text, axis=1)
evidence["evidence_len"] = evidence["evidence_text"].str.len()
evidence["caption_len"] = evidence["caption"].fillna("").astype(str).str.len()
evidence.to_csv(EVIDENCE_DIR / "raw_evidence_all.csv", index=False)
pickle.dump(evidence, open(EVIDENCE_DIR / "raw_evidence_all.pkl", "wb"), protocol=4)
for (dataset, split), sub in evidence.groupby(["dataset", "split"]):
    sub.to_csv(EVIDENCE_DIR / f"raw_evidence_{dataset}_{split}.csv", index=False)
display(evidence.head())


,pair_id,dataset,split,label,caption,evidence_text,evidence_len,caption_len
0,nc_620701_0,newsclippings,train,0,Saudi troops cheer as they ride at the back of...,Caption: Saudi troops cheer as they ride at th...,387,125
1,nc_620701_1,newsclippings,train,1,Israeli soldiers ride on a tank to a position ...,Caption: Israeli soldiers ride on a tank to a ...,328,73
2,nc_1569820_0,newsclippings,train,0,Tiffany is north east of Aberdeen,Caption: Tiffany is north east of Aberdeen | A...,325,33
3,nc_1569820_1,newsclippings,train,1,Some ferry services were affected between Holy...,Caption: Some ferry services were affected bet...,358,71
4,nc_1490080_0,newsclippings,train,0,Rock band Paramore is another group which sell...,Caption: Rock band Paramore is another group w...,349,55


In [5]:
summary = evidence.groupby(["dataset", "split"]).agg(rows=("pair_id", "count"), mean_evidence_len=("evidence_len", "mean"), mean_caption_len=("caption_len", "mean")).reset_index()
summary.to_csv(TABLES / "stream_c_evidence_summary.csv", index=False)
display(summary)
print("Evidence builder complete.")


,dataset,split,rows,mean_evidence_len,mean_caption_len
0,mmfakebench,test,10000,320.790400,79.623900
1,mmfakebench,val,1000,314.383000,79.030000
2,newsclippings,test,7264,389.938601,97.778634
3,newsclippings,train,71072,391.876477,98.137748
4,newsclippings,val,7024,388.852790,97.401054
5,verite,test,1001,222.350649,125.358641


Evidence builder complete.
